# OSSpRe 修复验证 notebook (Colab / Linux, renv 环境恢复)

本 notebook 在 Linux 上验证 OSSpRe 代码审查后的全部修复。环境完全按仓库自带的
`renv.lock` 恢复(R 4.6.1,179 个包的精确版本;`opentimsr`/`pmd`/`rmwf` 按
锁文件记录从 GitHub 恢复),保证与作者本地环境一致。

流程:

1. 安装系统库 + Posit r-builds 的 **R 4.6.1**(与 `renv.lock` 完全一致)
2. `renv::restore()` 按锁文件恢复全部 R 包(CRAN 走 P3M 二进制加速,
   Bioconductor 3.23 的 42 个包从源码编译,**这是全程最慢的一步**);
   `opentimsr` 从作者 drat 仓库(yufree.github.io/drat)恢复——上游 GitHub
   仓库是 monorepo,包内的 `opentims_core` symlink 会让 tarball 类安装器
   (renv)编译失败,drat 上是已 vendor 的版本,任何平台可直接安装
3. 从 GitHub 克隆仓库(main 分支,修复已合入),验证标记并兜底 renv.lock 的 drat 记录
4. 编译 4 个 C++ 文件(验证 OpenMP 保护/跨平台编译)
5. `peakalign.cpp` 二分重写:与暴力参考实现逐行对比 + 边缘情况 + 性能
6. `peak_finder.cpp`:合成数据种峰回收 + OpenMP 并发确定性(竞态回归测试)
7. Bruker `libtimsdata.so` 真实库:1/k0 ↔ CCS 往返转换 + dlopen 缓存
8. `helper.r` 端到端:PCA 分割维度、np 计数、ROI 聚类 NaN 防护、注释、空网络保护等
9. 输出 PASS/FAIL/SKIP 汇总表

**使用方法**: 直接在 Colab 打开本文件运行:
`https://colab.research.google.com/github/yufree/OSSpRe/blob/main/OSSpRe_colab_test.ipynb`
(或下载后 `文件 → 上传笔记本`),`代码执行程序 → 全部运行`。
默认 CPU 运行时即可。**全程约 40–60 分钟**(主要耗时在第 2 步 Bioconductor 源码包
编译),期间请保持标签页打开避免断连。

In [ ]:
%%bash
# ---- 系统库 + renv.lock 指定的 R 4.6.1 (Posit r-builds, 约 3 分钟) ----
set -e
apt-get update -qq
PKGS="build-essential gfortran git curl ca-certificates \
  libcurl4-openssl-dev libssl-dev libxml2-dev libzstd-dev \
  libfontconfig1-dev libfreetype6-dev pkg-config libpng-dev libtiff-dev \
  libjpeg-dev libnetcdf-dev libgit2-dev libglpk-dev libharfbuzz-dev \
  libfribidi-dev libcairo2-dev libxt-dev libblas-dev liblapack-dev"
DEBIAN_FRONTEND=noninteractive apt-get install -y --no-install-recommends $PKGS \
  > /tmp/apt.log 2>&1 || { echo "部分系统包安装失败, 继续尝试..."; tail -5 /tmp/apt.log; }

. /etc/os-release
case "$VERSION_CODENAME" in
  noble) UB=2404 ;;
  *)     UB=2204 ;;
esac
echo "$VERSION_CODENAME" > /content/codename
echo "Ubuntu $VERSION_CODENAME -> ubuntu-$UB"
if curl -fsSL -o /tmp/r.deb "https://cdn.posit.co/r/ubuntu-${UB}/pkgs/r-4.6.1_1_amd64.deb" \
   && DEBIAN_FRONTEND=noninteractive apt-get install -y /tmp/r.deb >> /tmp/apt.log 2>&1; then
  ln -sf /opt/R/4.6.1/bin/R /usr/local/bin/R
  ln -sf /opt/R/4.6.1/bin/Rscript /usr/local/bin/Rscript
  echo "已安装 r-builds R 4.6.1 (与 renv.lock 一致)"
else
  echo "警告: r-builds 安装失败, 回退 apt r-base (版本可能与锁文件不符)"
  DEBIAN_FRONTEND=noninteractive apt-get install -y r-base r-base-dev >> /tmp/apt.log 2>&1
fi
R --version | head -1
nproc

In [ ]:
# rpy2: 在 Python notebook 中运行 R 单元 (%%R)
# R 已通过 /usr/local/bin/R 链接到 /opt/R/4.6.1, rpy2 会基于它编译
!pip install -q rpy2

In [ ]:
%load_ext rpy2.ipython

## 1. 克隆仓库并验证修复已包含

修复已合入 GitHub main(≥ `79cb81b` 代码修复、`db86bf8` drat 环境修复)。
克隆后验证标记存在;若 `renv.lock` 仍是旧版(理论上不会),自动切换 opentimsr 到 drat。

In [ ]:
%%bash
set -e
cd /content
rm -rf OSSpRe
git clone --depth 1 https://github.com/yufree/OSSpRe.git
cd OSSpRe
git log --oneline -1
# 验证修复标记
grep -q "roi_idx" helper.r \
  && echo "代码修复已包含 (>= 79cb81b)" \
  || { echo "错误: 克隆的仓库缺少修复"; exit 1; }
# 确保 renv.lock 的 opentimsr 走 drat (正常情况下已包含, 此处为兜底)
grep -q 'yufree.github.io/drat' renv.lock || python3 - <<'PYEOF'
import json
lock = json.load(open('renv.lock'))
p = lock['Packages'].get('opentimsr')
if p:
    lock['Packages']['opentimsr'] = {k: v for k, v in p.items()
                                     if not k.startswith(('Remote', 'Github'))}
    lock['Packages']['opentimsr']['Source'] = 'Repository'
    repos = lock['R']['Repositories']
    if not any(r['Name'] == 'drat' for r in repos):
        repos.append({'Name': 'drat', 'URL': 'https://yufree.github.io/drat'})
    json.dump(lock, open('renv.lock', 'w'), indent=2, ensure_ascii=False)
    print('renv.lock: opentimsr 记录已切换到 drat')
PYEOF
echo "克隆就绪

## 2. `renv::restore()` 按锁文件恢复 R 环境 (约 30–50 分钟)

- R 版本: 4.6.1(与 `renv.lock` 一致)
- 178 个包:134 个 CRAN 包走 Posit P3M **二进制**(分钟级);42 个 Bioconductor 3.23
  包无二进制可用,从**源码编译**(mzR/MSnbase/xcms 较慢,全程最耗时的一步)
- `pmd` / `rmwf` 按锁文件 Remote 字段从 **GitHub** tarball 恢复(已验证可行)
- **`opentimsr` 从 drat 仓库恢复**(https://yufree.github.io/drat):上游
  `michalsta/opentims` 是 monorepo,R 包内的 `opentims_core` 是指向
  `src/opentims++` 的 symlink,renv 的 tarball 安装路径会因此编译失败;
  drat 上发布了把 symlink 替换为实体目录的 vendored 版本(commit `bbf5126`),
  已实测任何平台可直接 `renv::restore()`
- 若个别包版本已在 CRAN 更新,自动回退 cloud.r-project.org 的 Archive 归档

In [ ]:
%%R
# 仓库优先级: P3M 二进制 > drat > CRAN 源(+Archive) > Bioconductor 3.23 源
codename <- tryCatch(readLines("/content/codename", warn = FALSE), error = function(e) "jammy")
repos <- c(P3Mbin  = sprintf("https://packagemanager.posit.co/all/__linux__/%s/latest", codename),
           drat    = "https://yufree.github.io/drat",
           CRAN    = "https://cloud.r-project.org",
           BioCsoft = "https://bioconductor.org/packages/3.23/bioc")
options(repos = repos)
install.packages("renv", repos = repos[1])   # 二进制装 renv 本身

setwd("/content/OSSpRe")
source("renv/activate.R")                    # 激活项目库 (后续 %%R 单元继承)
cat("R:", getRversion(), "| 项目库:", .libPaths()[1], "\n")

t0 <- Sys.time()
renv::restore(prompt = FALSE, repos = repos)
cat(sprintf("renv::restore 完成, 耗时 %.1f 分钟\n",
            as.numeric(difftime(Sys.time(), t0, units = "mins"))))

need <- c("Rcpp", "data.table", "opentimsr", "pmd", "rmwf", "enviGCMS", "irlba", "ClusterR")
got <- sapply(need, requireNamespace, quietly = TRUE)
print(got)
stopifnot(all(got))
cat("环境就绪\n")

## 3. 编译 4 个 C++ 文件

验证 `#ifdef _OPENMP` 保护等跨平台修复在 Linux/gcc 下正常。

In [ ]:
%%R
setwd("/content/OSSpRe")
RESULTS <- new.env()
check <- function(name, expr) {
  ok <- tryCatch(isTRUE(force(expr)),
                 error = function(e) { cat("  ERROR:", conditionMessage(e), "\n"); FALSE })
  RESULTS[[name]] <- ok
  cat(sprintf("[%s] %s\n", if (ok) "PASS" else "FAIL", name))
  invisible(ok)
}
check("cpp_one_over_k0_compiles", { library(Rcpp); sourceCpp("one_over_k0_to_ccs.cpp"); TRUE })
check("cpp_ccs_to_k0_compiles",   { sourceCpp("ccs_to_one_over_k0.cpp"); TRUE })
check("cpp_peakalign_compiles",   { sourceCpp("peakalign.cpp"); TRUE })
check("cpp_peakfinder_compiles",  { sourceCpp("peak_finder.cpp"); TRUE })

## 4. `findpeakalign` 二分重写:正确性 + 性能

与旧算法逻辑等价的 R 暴力参考实现逐行对比(帧、键、强度),并测试空数据/零匹配边缘情况与规模扩展。

In [ ]:
%%R
set.seed(123)
n <- 4000
frame <- sample(1:3, n, replace = TRUE)
mz <- round(runif(n, 700, 800), 4)
ccs <- round(runif(n, 250, 320), 2)
intensity <- round(runif(n) * 1000, 1)
ref_mz <- c(710.05, 725.5, 733.33, 748.9, 760.02, 775.6, 790.123, 799.999)
ref_ccs <- c(280.1, 290.5, 300.2, 265.0, 310.3, 275.8, 295.4, 305.9)
ppm <- 20; ccs_shift <- 0.03

# R 暴力参考实现(与旧 C++ 的 O(N_ref x N_raw) 逻辑逐条对应)
ref_impl <- function() {
  out <- data.frame(frame = integer(), mz_ccs = character(), intensity = double())
  for (i in seq_along(ref_mz)) {
    mz_tol <- ppm * ref_mz[i] / 1e6; ccs_tol <- ccs_shift * ref_ccs[i]
    key <- paste0(sprintf("%.6f", ref_mz[i]), "_", sprintf("%.6f", ref_ccs[i]))
    for (f in sort(unique(frame))) {
      j <- which(frame == f)
      hit <- abs(mz[j] - ref_mz[i]) <= mz_tol & abs(ccs[j] - ref_ccs[i]) <= ccs_tol
      tot <- sum(intensity[j][hit])
      if (tot > 0) out <- rbind(out, data.frame(frame = f, mz_ccs = key, intensity = tot))
    }
  }
  out
}
r_new <- as.data.frame(findpeakalign(mz, ccs, intensity, frame, ref_mz, ref_ccs, ppm, ccs_shift))
r_ref <- ref_impl()
a <- r_new[order(r_new$frame, r_new$mz_ccs), ]; b <- r_ref[order(r_ref$frame, r_ref$mz_ccs), ]
check("align_matches_reference", nrow(a) == nrow(b) && identical(a$frame, b$frame) &&
        identical(a$mz_ccs, b$mz_ccs) && max(abs(a$intensity - b$intensity)) == 0)

e <- findpeakalign(numeric(0), numeric(0), numeric(0), integer(0), ref_mz, ref_ccs, ppm, ccs_shift)
z <- findpeakalign(mz, ccs, intensity, frame, 999.9, 200, ppm, ccs_shift)
check("align_edgecases", nrow(e) == 0 && nrow(z) == 0)

# 性能: 新实现多规模扩展 (旧算法为 O(N_ref x N_raw), R 参考实现作复杂度代理)
for (nn in c(2e4, 1e5, 5e5)) {
  mz2 <- round(runif(nn, 700, 800), 4); ccs2 <- round(runif(nn, 250, 320), 2)
  int2 <- runif(nn); fr2 <- sample(1:500, nn, replace = TRUE)
  rmz <- round(runif(500, 700, 800), 4); rccs <- runif(500, 250, 320)
  t0 <- Sys.time()
  res <- findpeakalign(mz2, ccs2, int2, fr2, rmz, rccs, ppm, ccs_shift)
  cat(sprintf("  %d 点 x 500 参考峰 x 500 帧: %.3f 秒 (%d 行命中)\n",
              nn, as.numeric(difftime(Sys.time(), t0, units = "secs")), nrow(res)))
}
t0 <- Sys.time(); invisible(ref_impl())
cat(sprintf("  [对照] R 暴力参考实现仅 %d 点 x 8 峰 x 3 帧就耗时 %.1f 秒\n",
            n, as.numeric(difftime(Sys.time(), t0, units = "secs"))))
check("align_benchmark_runs", TRUE)

## 5. `find_2d_peaks`:种峰回收 + 并发确定性

合成 m/z–CCS 平面上种植 5 个已知峰,验证能被找回;重复运行 5 次结果完全一致(验证 OpenMP 竞态修复)。

In [ ]:
%%R
set.seed(1)
n_bg <- 5000
mz <- runif(n_bg, 700, 800); ccs <- runif(n_bg, 250, 320); inten <- runif(n_bg, 1, 100)
centers <- data.frame(mz = seq(720, 780, length.out = 5), ccs = seq(260, 300, length.out = 5))
for (i in 1:5) {
  mz <- c(mz, centers$mz[i] + rnorm(30, 0, centers$mz[i] * 5e-6))
  ccs <- c(ccs, centers$ccs[i] + rnorm(30, 0, centers$ccs[i] * 0.005))
  inten <- c(inten, c(10000, runif(29, 5000, 9999)))
}
res <- find_2d_peaks(mz, ccs, inten, mz_ppm = 20, ccs_tolerance = 0.03, snr = 3, mz_bins = 1000, ccs_bins = 50)
found <- sapply(1:5, function(i)
  any(abs(res$mz - centers$mz[i]) <= centers$mz[i] * 50e-6 &
      abs(res$ccs - centers$ccs[i]) <= centers$ccs[i] * 0.05))
cat("  找到峰数:", nrow(res), " 5个种峰全部回收:", all(found), "\n")
check("peakfinder_recovers_planted", nrow(res) >= 5 && all(found))

r <- replicate(5, find_2d_peaks(mz, ccs, inten, 20, 0.03, 3, 1000, 50), simplify = FALSE)
check("peakfinder_deterministic_openmp", all(sapply(r[-1], function(x) identical(x, r[[1]]))))

# 退化输入: 单一 mz/ccs 值(旧版 Grid2D 除零)
d <- find_2d_peaks(rep(700.5, 10), rep(280, 10), 1:10, 20, 0.03, 3, 100, 50)
check("peakfinder_degenerate_input", nrow(d) >= 1 && !any(is.na(d$ccs)))

## 6. Bruker `libtimsdata.so` 真实库测试(可选)

下载 Bruker 库后测试 1/k0 → CCS → 1/k0 往返一致性,以及连续调用(dlopen 会话缓存)。下载失败时自动 SKIP。

In [ ]:
%%R
lib <- tryCatch(opentimsr::download_bruker_proprietary_code("/content/bruker"),
               error = function(e) NULL)
if (is.null(lib)) {
  RESULTS[["bruker_roundtrip"]] <- NA
  cat("[SKIP] bruker_roundtrip (Bruker 库下载不可用)\n")
} else {
  cat("Bruker 库:", lib, "\n")
  check("bruker_roundtrip", {
    k0 <- seq(0.6, 1.6, length.out = 50); mzv <- seq(200, 1000, length.out = 50)
    ccs <- one_over_k0_to_ccs_parallel(k0, rep(1, 50), mzv, lib)
    k0b <- ccs_to_one_over_k0_parallel(ccs, rep(1, 50), mzv, lib)
    max(abs(k0 - k0b)) < 1e-4 &&
      is.finite(sum(one_over_k0_to_ccs_parallel(k0, rep(1, 50), mzv, lib)))  # 二次调用走缓存
  })
}

## 7. `helper.r` 端到端测试

合成 ticmzccs.csv(含空间结构 + 全零离子),验证全部修复的 R 函数。

In [ ]:
%%R
t0 <- Sys.time()
suppressPackageStartupMessages(source("helper.r"))
cat(sprintf("helper.r 加载完成 (%.1fs)\n", as.numeric(difftime(Sys.time(), t0, units = "secs"))))

In [ ]:
%%R
set.seed(42)
n_pix <- 200; n_ion <- 60
xs <- rep(1:20, each = 10) * 10; ys <- rep(1:10, 20) * 10
mzv <- round(seq(700, 900, length.out = n_ion), 4)
imv <- round(seq(250, 320, length.out = n_ion), 2)
M <- sapply(seq_len(n_ion), function(i) {
  if (i <= 20) as.numeric(xs + rnorm(n_pix, 0, .3))
  else if (i <= 40) as.numeric(ys + rnorm(n_pix, 0, .3))
  else rep(0, n_pix)                       # 20 个全零离子
})
dt <- data.table(location = paste0(xs, "_", ys), M)
colnames(dt) <- c("location", paste0(mzv, "_", imv))
csv <- "test_ticmzccs.csv"; fwrite(dt, csv)

# getsummary
s <- capture.output(getsummary(csv))
check("helper_getsummary", any(grepl("m/z range", s)))

# np 计数不再多算 1
invisible(capture.output(plot_peak_stats(csv)))
d2 <- fread(csv)
np_fn <- d2[, rowSums(.SD != 0), .SDcols = -1]
np_manual <- rowSums(as.matrix(d2[, -1]) != 0)
check("helper_np_counts_correct", isTRUE(all.equal(np_fn, np_manual)))

# PCA 分割聚类的是像素
invisible(capture.output(perform_pca_segmentation(csv, "test_seg.csv")))
seg <- fread("test_seg.csv")
check("helper_pca_clusters_pixels", nrow(seg) == n_pix && length(unique(seg$seg)) >= 2)

# reactomics 列结构
perform_reactomics_analysis(csv, "test_react.csv")
rr <- fread("test_react.csv")
check("helper_reactomics_structure", nrow(rr) == n_pix && ncol(rr) == 2 + 3 * 4)

# ROI 聚类: 后半 ROI 中 20 个全零离子(旧版 NaN 让 hclust 崩溃)
locindex <- rep(c(TRUE, FALSE), each = n_pix / 2)
okc <- tryCatch({
  suppressWarnings(perform_cluster_ions(csv, "test_ioncluster.csv", locindex = locindex,
                                        hclust_cutoff = 0.6, min_cluster_size = 3,
                                        folder_name = "test_cluster"))
  TRUE
}, error = function(e) { cat("  错误:", conditionMessage(e), "\n"); FALSE })
ic <- if (file.exists("test_ioncluster.csv")) fread("test_ioncluster.csv") else data.table()
check("helper_cluster_roi_no_nan_crash", okc && nrow(ic) == n_ion)

# getanno: 用数据库精确值构造列名
sel <- fread("lipidall.csv")[adducts %in% c("[M+H]", "[M+Na]")][1:5]
dt6 <- data.table(location = paste0(xs, "_", ys), matrix(runif(n_pix * 5), nrow = n_pix))
colnames(dt6) <- c("location", paste0(sel$mz, "_", sel$ccs))
fwrite(dt6, "test_peak5.csv")
getanno(database = "lipidall.csv", mode = "pos", peakpath = "test_peak5.csv", annofile = "test_anno.csv")
check("helper_getanno", nrow(fread("test_anno.csv")) >= 5)

# 空注释网络保护
fwrite(data.frame(name = character(), mz = numeric()), "test_emptyanno.csv")
okn <- tryCatch({
  generate_molecular_network(csv, "test_ioncluster.csv", "test_emptyanno.csv", "test_net.csv"); TRUE
}, error = function(e) FALSE)
check("helper_network_empty_guard", okn && nrow(fread("test_net.csv")) == 0)

# 签名修复
fg <- formals(getqlist); fr <- formals(getrefpeak)
check("helper_signatures", all(c("mz_ppm", "ccs_tolerance") %in% names(fg)) &&
        identical(fr$libpath, "./") && !exists("perform_roi_reactomics_analysis"))

## 8. 汇总报告

In [ ]:
%%R
sm <- data.frame(test = names(RESULTS), pass = unlist(RESULTS, use.names = FALSE))
print(sm, row.names = FALSE)
cat(sprintf("\n%d PASS / %d FAIL / %d SKIP\n",
            sum(sm$pass, na.rm = TRUE), sum(!sm$pass, na.rm = TRUE), sum(is.na(sm$pass))))
write.csv(sm, "/content/test_summary.csv", row.names = FALSE)

In [ ]:
import pandas as pd
from IPython.display import Image, display
import os

s = pd.read_csv('/content/test_summary.csv')
def color(v):
    if pd.isna(v): return 'color: gray'
    return 'color: green' if v else 'color: red; font-weight: bold'
if hasattr(s.style, 'map'):
    display(s.style.map(color, subset=['pass']))
else:
    display(s.style.applymap(color, subset=['pass']))

# 展示 ROI 聚类生成的一张 cluster 图像(可视化确认)
for p in ['/content/OSSpRe/test_cluster']:
    if os.path.isdir(p):
        pngs = sorted(f for f in os.listdir(p) if f.endswith('.png'))
        if pngs:
            print('cluster 图像示例:', pngs[0])
            display(Image(os.path.join(p, pngs[0]), width=300))

## 结果解读

- **全部 PASS**:所有修复在 Linux + gcc/OpenMP + renv 恢复的作者环境 + 真实 Bruker 库下验证通过,可以放心提交/推送。
- **bruker_roundtrip SKIP**:Colab 无法访问 Bruker 下载源(网络策略),不影响其他结论;在本地 Linux/有网络的环境可手动验证。
- 任何 **FAIL**:展开对应单元格查看 `ERROR:` 输出,连同 `/content/test_summary.csv` 一起反馈。
- 若 `renv::restore()` 个别包失败:通常是网络抖动,`代码执行程序 → 运行单元格前` 重新运行第 2 节单元格即可续传(renv 只补缺的包)。

## 用真实数据测试(可选)

C++ 与算法层已被上面充分覆盖;`getrefpeak`/`getqlist` 需要真实 Bruker `.d` 数据,可在本 notebook 末尾追加单元格:

```r
# 将 raw.d 打包为 zip 上传到 /content 后:
system("cd /content && unzip -q raw.zip")
path <- "/content/raw.d"
getrefpeak(path = path, accept_Bruker_EULA_and_on_Windows_or_Linux = TRUE,
           libpath = "/content/bruker/", batch_size = 100000,
           outref = "refpeak.csv", outcoord = "coord.csv",
           xrange = c(0, 1e9), yrange = c(0, 1e9))
# ...然后按 workflow.Rmd 的步骤继续
```

注意 `workflow.Rmd` 默认 `eval = FALSE`(教程模式),真实数据测试请逐 chunk 运行或将 setup 改为 `eval = TRUE`。